# 02 — LCEL Chains

Companion notebook to `02-langchain-and-lcel.md`.

Demonstrates the `prompt | model | output_parser` pipe-composition pattern using LangChain's
**fake chat model** (`FakeListChatModel` / `FakeMessagesListChatModel`), so it runs fully offline --
no Azure OpenAI credentials needed. The chain shape here is identical to what you'd write against a
real `AzureChatOpenAI` model; only the model object changes.

Requires: `langchain-core` (the fake models below have shipped in `langchain_core` for a while;
if your installed version differs slightly, see the fallback cell).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

try:
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
except ImportError:
    # older/newer langchain_core versions sometimes expose this from a different path
    from langchain_core.language_models.fake import FakeListLLM as FakeListChatModel

print("Imports OK")

## Build the pieces

Same three pieces described in Chapter 2: a `ChatPromptTemplate`, a chat-model-shaped `Runnable`
(here, a fake one that returns pre-scripted responses instead of calling a real API), and an output
parser.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an internal assistant for {client_name}. Only answer using the provided context."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

# FakeListChatModel cycles through this fixed list of responses on each call --
# a drop-in stand-in for AzureChatOpenAI(...) with zero network calls.
fake_model = FakeListChatModel(responses=[
    "Refunds are processed within 5-7 business days, per the Refund Policy v3.",
])

parser = StrOutputParser()

## Compose with LCEL's `|` operator

This is the exact pattern from `02-langchain-and-lcel.md`: `prompt | model | parser`. Every piece
implements the same `Runnable` interface, so `|` chains them into a single `RunnableSequence`.

In [ ]:
chain = prompt | fake_model | parser

response = chain.invoke({
    "client_name": "Acme Bank",
    "context": "Refunds are processed within 5-7 business days.",
    "question": "How long do refunds take?",
})
print(response)

## Streaming

`chain.stream(...)` yields output incrementally. With a real chat model this yields tokens as
they're generated; `FakeListChatModel` doesn't truly stream token-by-token, but it still
demonstrates the *interface* -- the exact same loop shape you'd use with `AzureChatOpenAI`, which
is what Chapter 4's Streamlit/Chainlit UI code iterates over to render tokens as they arrive.

In [ ]:
print("Streaming response:")
for chunk in chain.stream({
    "client_name": "Acme Bank",
    "context": "Refunds are processed within 5-7 business days.",
    "question": "How long do refunds take?",
}):
    print(chunk, end="", flush=True)
print()

## Batching

`chain.batch([...])` runs multiple inputs through the chain, useful for offline evaluation runs
(e.g., running a held-out set of Q&A pairs through the chain to check for regressions after a
prompt change -- see the evaluation discussion in `99-Interview-QA.md`, question 7).

In [ ]:
# a fake model with more scripted responses so batch has something distinct to return per item
batch_model = FakeListChatModel(responses=[
    "Refunds are processed within 5-7 business days.",
    "Branches are open 9am-5pm, Monday to Friday.",
    "You can reset your password from the account settings page.",
])
batch_chain = prompt | batch_model | parser

inputs = [
    {"client_name": "Acme Bank", "context": "Refunds take 5-7 business days.", "question": "How long do refunds take?"},
    {"client_name": "Acme Bank", "context": "Branches: 9-5 Mon-Fri.", "question": "What are your branch hours?"},
    {"client_name": "Acme Bank", "context": "Password reset via account settings.", "question": "How do I reset my password?"},
]

results = batch_chain.batch(inputs)
for q, r in zip(inputs, results):
    print(f"Q: {q['question']}\nA: {r}\n")

## Swapping in a real Azure OpenAI model

Because every piece here is a `Runnable`, swapping the fake model for the real one is a one-line
change -- nothing else in the chain needs to change:

```python
from langchain_openai import AzureChatOpenAI

real_model = AzureChatOpenAI(
    azure_deployment="<your-deployment-name>",
    api_version="2024-05-01-preview",
    temperature=0.2,
)

chain = prompt | real_model | parser   # identical shape, real model
```

Requires `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT` (or equivalent) environment variables.
See `03-chatbot-architecture-azure-openai.md` for deployment/rate-limit details.